# CodeTune v2 — 06 DPO Data Generation (Execution-Driven)

**Environment**: Google Colab A100 80GB  
**Model**: `Michlitt/codetune-v2-sft-C`（bf16，加载约 18GB VRAM）  
**Output**: `data/processed/dpo_pairs_exec.jsonl` + `dpo_pairs_exec_val.jsonl`  
**预计耗时**: ~25 分钟

## 两种配对方法（单次推理完成）

| 方法 | 信号来源 | chosen | rejected |
|------|----------|--------|----------|
| **执行驱动** | HumanEval 164题 × 4温度 | 通过测试的候选 | 失败的候选 |
| **SFT失败兜底** | 全部候选均失败时 | canonical_solution（验证通过） | temp=0.2 失败输出 |

## 运行顺序
| Cell | 任务 | 预计时间 |
|------|------|----------|
| 1 | GPU 检查 | 即时 |
| 2 | 安装依赖 | ~3 min |
| 3 | HF 登录 + Drive 挂载 | ~1 min |
| 4 | 配置 | 即时 |
| 5 | 下载 sft_C + 加载模型 + 定义工具函数 | ~5 min |
| 6 | 加载 HumanEval | 即时 |
| 7 | 主循环（Method 1 + 2 一次推理） | ~15 min |
| 8 | 合并、去重、统计 | 即时 |
| 9 | 分割、保存 + Drive 备份 | 即时 |
| 10 | 上传到 HF Hub dataset | ~1 min |
| 11 | 质量预览 | 即时 |

In [ ]:
# Cell 1 — GPU 检查
import subprocess, torch
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU:", r.stdout.strip())
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB, "
      f"BF16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Cell 2 — 安装（unsloth 必须最先安装）
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes -q
!pip install datasets huggingface-hub -q

# Stub trl 依赖的可选包，避免 ImportError
import sys, types
from importlib.machinery import ModuleSpec

_STUBS = [
    "llm_blender", "llm_blender.blender", "llm_blender.blender.blender_utils",
    "llm_blender.pair_ranker", "weave", "weave.trace",
    "weave.trace.context", "weave.trace.context.weave_client_context",
]
for _name in _STUBS:
    m = types.ModuleType(_name)
    m.__spec__ = ModuleSpec(_name, None)
    m.__file__ = None
    m.__path__ = []
    m.__package__ = _name.split(".")[0]
    sys.modules[_name] = m

sys.modules["weave"].EvaluationLogger = type("EvaluationLogger", (), {})
sys.modules["weave.trace.context.weave_client_context"].get_weave_client = lambda: None

print("Done.")

In [ ]:
# Cell 3 — HF 登录 + Drive 挂载
from huggingface_hub import login
from google.colab import drive
import os

HF_TOKEN    = "YOUR_HF_TOKEN"
HF_USERNAME = "Michlitt"

login(token=HF_TOKEN)
print("Logged in as", HF_USERNAME)

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/codetune"
print(f"Drive mounted. DRIVE_ROOT={DRIVE_ROOT}")

In [ ]:
# Cell 4 — 配置
from pathlib import Path

SFT_CHECKPOINT = f"{HF_USERNAME}/codetune-v2-sft-C"
CHECKPOINT_DIR = Path("sft_C")

OUTPUT_TRAIN = Path("data/processed/dpo_pairs_exec.jsonl")
OUTPUT_VAL   = Path("data/processed/dpo_pairs_exec_val.jsonl")
DRIVE_BACKUP = Path(DRIVE_ROOT) / "dpo_exec_pairs"

TEMPERATURES   = [0.2, 0.6, 1.0, 1.3]   # 每题生成 4 个候选
MAX_NEW_TOKENS = 512
TEST_TIMEOUT   = 10                       # 每题 subprocess 超时（秒）
VAL_RATIO      = 0.1
SEED           = 42

SYSTEM_PROMPT = (
    "You are an expert Python programmer. Complete the given function. "
    "Write only the function body — no extra explanation, no test code."
)

print("Config loaded.")
print(f"  SFT checkpoint : {SFT_CHECKPOINT}")
print(f"  Output train   : {OUTPUT_TRAIN}")
print(f"  Temperatures   : {TEMPERATURES}")

In [ ]:
# Cell 5 — 下载 sft_C + 加载模型（bf16）+ 定义工具函数
# 预计耗时：~5 分钟
import sys, subprocess, tempfile, torch
from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel

# ── 下载模型 ──────────────────────────────────────────────────────────────
CHECKPOINT_DIR.mkdir(exist_ok=True)
if not (CHECKPOINT_DIR / "adapter_config.json").exists():
    print(f"Downloading {SFT_CHECKPOINT} ...")
    snapshot_download(
        repo_id=SFT_CHECKPOINT,
        local_dir=str(CHECKPOINT_DIR),
        repo_type="model",
    )
    print("Download done.")
else:
    print(f"Already downloaded: {CHECKPOINT_DIR}")

# ── 加载模型（A100 80GB，bf16 更快） ──────────────────────────────────────
print("Loading model (bf16) ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(CHECKPOINT_DIR),
    max_seq_length=1536,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"   # decoder-only 模型必须左 padding
used = torch.cuda.memory_allocated(0) / 1e9
print(f"Model ready. VRAM used: {used:.1f} GB")


# ── 工具函数 ──────────────────────────────────────────────────────────────
def format_prompt(problem_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Complete this Python function:\n\n{problem_prompt}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def batch_generate(formatted_prompts: list[str], temperature: float, batch_size: int = 16) -> list[str]:
    """对所有 prompts 以同一 temperature 做 batch 推理，返回与输入等长的 completion 列表。"""
    do_sample = temperature > 0.0
    results = []
    for i in range(0, len(formatted_prompts), batch_size):
        chunk = formatted_prompts[i : i + batch_size]
        enc = tokenizer(
            text=chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
            padding=True,
        ).to(model.device)
        kwargs = dict(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )
        if do_sample:
            kwargs["temperature"] = temperature
            kwargs["top_p"] = 0.95
        with torch.no_grad():
            outs = model.generate(**kwargs)
        prompt_len = enc["input_ids"].shape[1]
        for out in outs:
            results.append(
                tokenizer.decode(out[prompt_len:], skip_special_tokens=True).strip()
            )
    return results


def run_tests(problem_prompt: str, completion: str,
              test_code: str, entry_point: str) -> dict:
    """用 subprocess 沙盒执行方案 + 测试，返回 {passed, stderr}。"""
    full_code = (
        problem_prompt + completion + "\n\n"
        + test_code + f"\n\ncheck({entry_point})\n"
    )
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False, encoding="utf-8"
    ) as f:
        f.write(full_code)
        tmp = f.name
    try:
        r = subprocess.run(
            [sys.executable, tmp],
            capture_output=True, text=True, timeout=TEST_TIMEOUT,
        )
        return {"passed": r.returncode == 0, "stderr": r.stderr[:300]}
    except subprocess.TimeoutExpired:
        return {"passed": False, "stderr": "TIMEOUT"}
    except Exception as e:
        return {"passed": False, "stderr": str(e)}
    finally:
        Path(tmp).unlink(missing_ok=True)


print("Helper functions defined.")

In [ ]:
# Cell 6 — 加载 HumanEval
from datasets import load_dataset

he_ds = load_dataset("openai/openai_humaneval", split="test", trust_remote_code=True)
print(f"HumanEval: {len(he_ds)} problems")

sample = he_ds[0]
print(f"\nSample: {sample['task_id']}")
print(f"Prompt (60 chars): {sample['prompt'][:60]}...")

In [ ]:
# Cell 7 — 主循环（batch 推理 + 并行测试）
# 预计耗时：~3-5 分钟生成 + ~30 秒测试
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Step 1: 预处理所有 prompt ─────────────────────────────────────────────
problems  = list(he_ds)
n         = len(problems)
formatted = [format_prompt(p["prompt"]) for p in problems]
print(f"Formatted {n} prompts.")

# ── Step 2: 按 temperature batch 生成（4 pass，每 pass 164 题）─────────────
all_candidates = {}
for t in TEMPERATURES:
    print(f"Generating temperature={t} ...")
    all_candidates[t] = batch_generate(formatted, temperature=t, batch_size=16)
print("Generation done.")

# ── Step 3: 并行执行测试 ──────────────────────────────────────────────────
# 每道题需要测试 4 个候选（+ 可能的 canonical），用线程池并行跑所有 subprocess
NUM_WORKERS = 32   # subprocess 是 I/O 密集，线程数可以开大

def test_problem(i):
    p           = problems[i]
    prompt      = p["prompt"]
    test_code   = p["test"]
    entry_point = p["entry_point"]
    canonical   = p["canonical_solution"]
    candidates  = [all_candidates[t][i] for t in TEMPERATURES]

    results = [run_tests(prompt, c, test_code, entry_point) for c in candidates]
    passing = [c for c, r in zip(candidates, results) if r["passed"]]
    failing = [c for c, r in zip(candidates, results) if not r["passed"]]
    return i, candidates, passing, failing, canonical, p["task_id"]

exec_pairs = []
sft_pairs  = []
skipped_all_pass   = 0
skipped_canon_fail = 0

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(test_problem, i): i for i in range(n)}
    for fut in tqdm(as_completed(futures), total=n, desc="Testing & pairing", unit="problem"):
        i, candidates, passing, failing, canonical, task_id = fut.result()

        if passing and failing:
            exec_pairs.append({
                "prompt":   formatted[i],
                "chosen":   passing[0],
                "rejected": failing[-1],
                "source":   "execution_driven",
                "task_id":  task_id,
            })
        elif not passing:
            r_canon = run_tests(problems[i]["prompt"], canonical,
                                problems[i]["test"], problems[i]["entry_point"])
            if r_canon["passed"]:
                sft_pairs.append({
                    "prompt":   formatted[i],
                    "chosen":   canonical,
                    "rejected": candidates[0],
                    "source":   "sft_failure",
                    "task_id":  task_id,
                })
            else:
                skipped_canon_fail += 1
        else:
            skipped_all_pass += 1

print(f"\n执行驱动对数  (Method 1): {len(exec_pairs)}")
print(f"SFT失败兜底对数 (Method 2): {len(sft_pairs)}")
print(f"跳过（全通过）            : {skipped_all_pass}")
print(f"跳过（canonical也失败）   : {skipped_canon_fail}")

In [ ]:
# Cell 8 — 合并、去重、统计
from collections import Counter

# exec_driven 优先；若同一 task_id 两种方法都有，保留 exec_driven
all_pairs = exec_pairs + sft_pairs
seen  = set()
deduped = []
for p in all_pairs:
    tid = p["task_id"]
    if tid not in seen:
        seen.add(tid)
        deduped.append(p)

removed = len(all_pairs) - len(deduped)
print(f"总对数     : {len(all_pairs)}")
print(f"去重后     : {len(deduped)}  （移除 {removed} 条重复）")
print()

src_counts = Counter(p["source"] for p in deduped)
for src, cnt in src_counts.most_common():
    print(f"  {src:<35} {cnt}")

In [ ]:
# Cell 9 — 分割、保存 + Drive 备份
import json, random, shutil

random.seed(SEED)
random.shuffle(deduped)

split       = max(1, int(len(deduped) * (1 - VAL_RATIO)))
train_pairs = deduped[:split]
val_pairs   = deduped[split:]

# 保存到 /content/data/processed/
OUTPUT_TRAIN.parent.mkdir(parents=True, exist_ok=True)
for path, data in [(OUTPUT_TRAIN, train_pairs), (OUTPUT_VAL, val_pairs)]:
    path.write_text(
        "\n".join(json.dumps(p, ensure_ascii=False) for p in data) + "\n",
        encoding="utf-8",
    )
    print(f"Saved: {path}  ({len(data)} pairs)")

# 备份到 Google Drive
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
for path in [OUTPUT_TRAIN, OUTPUT_VAL]:
    shutil.copy(path, DRIVE_BACKUP / path.name)
    print(f"Drive backup: {DRIVE_BACKUP / path.name}")

In [ ]:
# Cell 10 — 上传到 HF Hub dataset
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
DATASET_REPO = f"{HF_USERNAME}/codetune-v2-sft"

for path in [OUTPUT_TRAIN, OUTPUT_VAL]:
    api.upload_file(
        path_or_fileobj=str(path),
        path_in_repo=path.name,
        repo_id=DATASET_REPO,
        repo_type="dataset",
    )
    print(f"Uploaded: {path.name} → {DATASET_REPO}")

print(f"\nDone. Dataset: https://huggingface.co/datasets/{DATASET_REPO}")

In [ ]:
# Cell 11 — 质量预览（随机打印 3 对）
import textwrap

samples = random.sample(deduped, min(3, len(deduped)))

for i, pair in enumerate(samples):
    print("=" * 65)
    print(f"[{i+1}] task_id={pair['task_id']}  source={pair['source']}")

    # 从 prompt 中取出 user 部分（跳过 chat template 头）
    lines = pair["prompt"].split("\n")
    start = next(
        (j for j, l in enumerate(lines) if "Complete this Python function" in l), 0
    )
    print("PROMPT:")
    print("\n".join(lines[start : start + 6]))
    print()
    print("CHOSEN (300 chars):")
    print(pair["chosen"][:300])
    print()
    print("REJECTED (300 chars):")
    print(pair["rejected"][:300])
    print()

## 与现有 DPO pairs 合并

如果 `dpo_pairs.jsonl`（旧，来源 `generated` + `targeted_humaneval`）已存在，
可在 `03_dpo_training.ipynb` 中加一段合并代码：

```python
import json
from pathlib import Path

old  = [json.loads(l) for l in Path("data/processed/dpo_pairs.jsonl").read_text().splitlines() if l.strip()]
new  = [json.loads(l) for l in Path("data/processed/dpo_pairs_exec.jsonl").read_text().splitlines() if l.strip()]

# exec_driven 覆盖同 task_id 的旧对（exec 信号更可靠）
exec_ids = {p["task_id"] for p in new if p.get("source") == "execution_driven"}
old_filtered = [p for p in old if p.get("task_id") not in exec_ids]
merged = old_filtered + new

Path("data/processed/dpo_pairs_merged.jsonl").write_text(
    "\n".join(json.dumps(p, ensure_ascii=False) for p in merged) + "\n",
    encoding="utf-8",
)
print(f"Merged: {len(merged)} pairs (old={len(old_filtered)}, new={len(new)})")
```

然后在 `03_dpo_training.ipynb` 的 Cell 8 中将数据路径改为 `dpo_pairs_merged.jsonl`。